# Lab: Machine Learning for Anomalous Property Classification — Part 2

### Preamble

In Part 1, you built a classifier for the National Anomalous Property Register by applying each preprocessing step manually and in sequence: encoding categorical columns, assembling feature vectors, normalising, fitting a model. The result worked, but the steps were scattered across cells and tightly coupled to a particular DataFrame. Running the same preprocessing on new data — say, a fresh batch of field reports from the regional offices — would mean re-executing every cell in the right order and hoping nothing had changed in scope.

Part 2 introduces the **ML Pipeline**: a single object that encapsulates the entire preprocessing and training workflow. Fitting a pipeline on training data produces a `PipelineModel` that can be saved, loaded, and applied to new data with a single `.transform()` call. We also introduce **Word2Vec** for embedding the free-text columns, and **CrossValidator** for systematic hyperparameter search — both of which slot naturally into a pipeline. Finally, every model we train is tracked with **MLflow**, so hyperparameters, metrics and grid-search results are recorded systematically and can be compared across runs; the tuning section closes with a guided grid-search exercise.

## 1. Setup

In [0]:
!pip install pyspark matplotlib mlflow scikit-learn "nbformat>=4.2.0" --quiet

In [0]:
import math
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, size, concat_ws, split
from pyspark.sql.types import IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.feature import (StringIndexer, OneHotEncoder, VectorAssembler, 
                                Imputer, MinMaxScaler, Word2Vec, Tokenizer, PCA)
from pyspark.ml.clustering import KMeans
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import (
    ClusteringEvaluator, MulticlassClassificationEvaluator,
)
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.stat import Summarizer

import mlflow

spark = SparkSession.builder.getOrCreate()


In [0]:
import os

SPARKML_TEMP_PATH = "/Volumes/workspace/default/bda_week11/sparkml_temp"
dbutils.fs.mkdirs(SPARKML_TEMP_PATH)
os.environ["SPARKML_TEMP_DFS_PATH"] = SPARKML_TEMP_PATH

## 2. Loading the Dataset

We load the same NAPR Parquet file from Part 1 and apply the same light preprocessing — null drop, derived features, boolean and target casting — in a single consolidated block. This is boilerplate you have already seen; run it and move on.

In [0]:
properties = spark.read.parquet("/Workspace/Users/dcaridade@novaims.unl.pt/BigDataAnalysis2627/weeks/week10/haunted_houses.parquet")
properties.count(), len(properties.columns)

In [0]:
# Null drop, derived features, cast target — identical to Part 1
properties = properties.na.drop()

CURRENT_YEAR = 2026
properties = properties \
    .withColumn("property_age", lit(CURRENT_YEAR) - col("construction_date")) \
    .withColumn("sightings_count", size(split(col("sightings"), r"\|")))

bool_cols = ["warning", "teenage_group_disappearance", "foggy", "belltower", "moor_nearby"]
for c in bool_cols:
    properties = properties.withColumn(c, col(c).cast(IntegerType()))
properties = properties.withColumn("haunted", col("haunted").cast(IntegerType()))

TARGET = "haunted"

properties.count()  # records remaining after null drop


## 3. Transformers and Estimators

Before building a pipeline, it helps to be precise about the two kinds of objects it can contain.

**Transformer** — has only a `transform(df)` method. It applies a fixed, data-independent operation: `VectorAssembler`, `OneHotEncoder` (after fitting), `Tokenizer`. No learning happens; the same logic applies to any DataFrame.

**Estimator** — has a `fit(df)` method that *learns* something from the data and returns a fitted **transformer** (a `Model`). Examples: `StringIndexer`, `Imputer`, `MinMaxScaler`, `Word2Vec`, `LogisticRegression`. You must call `.fit()` before you can call `.transform()`.

The distinction matters because a `Pipeline` handles them differently under the hood: for each stage it calls `fit()` if the stage is an estimator, `transform()` if it is a transformer, passing the output of one stage as the input to the next.

### Data contamination

Estimators *learn from whatever data you pass to `.fit()`*. If you fit an `Imputer` or `MinMaxScaler` on the full dataset before splitting, the learned statistics (mean, min, max) will reflect test-set values too — the model has indirectly 'seen' the test set before evaluation. This is **data leakage**, and it produces optimistically biased metrics.

The correct discipline is: **fit only on training data, transform both splits**.

```python
# Wrong — leaks test statistics into the fitted parameters
imputer_model = imputer.fit(full_dataset)

# Correct — learns only from training data
imputer_model = imputer.fit(training)
train_imputed = imputer_model.transform(training)
test_imputed  = imputer_model.transform(test)
```

### Is there a `fit_transform()` like in scikit-learn?

No. Spark ML does not have `fit_transform()`, and the `Pipeline` object is the idiomatic replacement. Calling `pipeline.fit(training)` fits every estimator stage on the training data, producing a `PipelineModel`. That model is then a pure transformer: `pipeline_model.transform(test)` applies all stages — with the parameters learned from training — to the test set without refitting anything.

## 4. Building a Pipeline

A `Pipeline` is itself an estimator whose only parameter is `stages`: an ordered list of transformers and estimators. Calling `.fit()` on a Pipeline produces a `PipelineModel`, which is a transformer. That `PipelineModel` can then be applied to new data with a single `.transform()` call — the entire preprocessing chain runs automatically in the correct order.

This is the key practical advantage over Part 1's manual approach: the pipeline guarantees that test data and any future production data go through exactly the same transformations as the training data, fitted on the training data only.

### 4.1 Preprocessing stages

We define each preprocessing stage as a named variable. The boilerplate (column lists, output column names) is filled in; fill in the missing estimator and transformer definitions.

**TODO:**
- Instantiate `StringIndexer` for `phenomenon_type` and `ghost`.
- Instantiate `OneHotEncoder` on the resulting index columns.
- Instantiate `Imputer` with `strategy="mean"` on the numeric columns.
- Instantiate `MinMaxScaler` to bring the assembled numeric vector to [0, 1].
- Instantiate the final `VectorAssembler` that combines scaled numerics, boolean flags, and OHE vectors into `"features"`.

In [0]:
# --- Categorical encoding ---
indexer = StringIndexer(
    inputCols     = ["phenomenon_type", "ghost"],
    outputCols    = ["phenomenon_idx", "ghost_idx"],
    handleInvalid = "keep",
)

encoder = OneHotEncoder(
    inputCols  = ["phenomenon_idx", "ghost_idx"],
    outputCols = ["phenomenon_vec", "ghost_vec"],
)

# --- Numeric imputation ---
numeric_cols = [
    "property_age", "sightings_count",
    "owl_hoots", "villagers_count", "disappeared_visitors",
    "dread", "tripadvisor_stars", "insurance_exclusion_clauses",
]
numeric_cols_imputed = [c + "_i" for c in numeric_cols]

imputer = Imputer(
    strategy   = "mean",
    inputCols  = numeric_cols,
    outputCols = numeric_cols_imputed,
)

# --- Scaling: bring all numeric features to [0, 1] ---
numeric_assembler = VectorAssembler(inputCols=numeric_cols_imputed, outputCol="numeric_raw")

scaler = MinMaxScaler(inputCol="numeric_raw", outputCol="numeric_scaled")

# --- Final feature assembly ---
feature_assembler = VectorAssembler(
    inputCols = ["numeric_scaled"] + bool_cols + ["phenomenon_vec", "ghost_vec"],
    outputCol = "features",
)

### 4.2 Assembling and fitting the pipeline

We pass the stages in order to `Pipeline`, then fit on the training set. Note that we split *before* fitting — the pipeline must never see the test set during training, including during the `Imputer` and `MinMaxScaler` fitting steps.

In [0]:
training, test = [df for df in properties.randomSplit([0.7, 0.3], seed=753)]
training.count(), test.count()  # trigger cache materialisation

In [0]:
# Concrete example: VectorAssembler is a transformer — no fit() needed
demo_assembler = VectorAssembler(
    inputCols=["owl_hoots", "dread", "tripadvisor_stars"],
    outputCol="demo_features",
)
demo_assembler.explainParam("outputCol")

# Imputer is an estimator — it must be fitted, and only on training data. Fitting on the full dataset would leak test-set statistics.
demo_imputer = Imputer(
    strategy   = "mean",
    inputCols  = ["owl_hoots", "dread"],
    outputCols = ["owl_hoots_i", "dread_i"],
)
demo_imputer_model = demo_imputer.fit(training)

# Show what the imputer learned: the mean of each column on the training set. These are the values that will fill nulls at transform time.
training.agg(
    {"owl_hoots": "mean", "dread": "mean"}
).show()


In [0]:
lr = LogisticRegression(featuresCol="features", labelCol=TARGET)

pipeline = Pipeline(stages=[
    indexer,
    encoder,
    imputer,
    numeric_assembler,
    scaler,
    feature_assembler,
    lr,
])

pipeline_model = pipeline.fit(training)

In [0]:
results = pipeline_model.transform(test)
results.select("prediction", "probability", TARGET).show(5, truncate=50)

### 4.3 Evaluating the pipeline model

We can extract individual fitted stages from the `PipelineModel` using `.stages`, which mirrors the order of stages passed to the pipeline.

In [0]:
# Native Spark evaluators — precision, recall and F1. All three are MulticlassClassificationEvaluator metrics computed natively by Spark: weightedPrecision / weightedRecall average the per-class scores weighted by class support, and f1 is their harmonic combination.
evaluator = MulticlassClassificationEvaluator(
    labelCol=TARGET, predictionCol="prediction", metricName="f1"
)
precision_evaluator = MulticlassClassificationEvaluator(
    labelCol=TARGET, predictionCol="prediction", metricName="weightedPrecision"
)
recall_evaluator = MulticlassClassificationEvaluator(
    labelCol=TARGET, predictionCol="prediction", metricName="weightedRecall"
)

f1_pipeline   = evaluator.evaluate(results)
prec_pipeline = precision_evaluator.evaluate(results)
rec_pipeline  = recall_evaluator.evaluate(results)

print(f"Pipeline LR — Test precision (native Spark): {prec_pipeline:.4f}")
print(f"Pipeline LR — Test recall    (native Spark): {rec_pipeline:.4f}")
print(f"Pipeline LR — Test F1        (native Spark): {f1_pipeline:.4f}")

# Extract the fitted LR model from the last stage (reused by MLflow in Section 6)
lr_model = pipeline_model.stages[-1]

In [0]:
# Confusion matrix
results.groupBy(TARGET).pivot("prediction").count().show()

### 4.4 F1: weighted (native) and macro (helper)

The confusion matrix above shows a sharp precision/recall asymmetry — the model misses roughly half of the truly haunted properties. **F1**, the harmonic mean of precision and recall, is the standard single-number summary of that trade-off, and Spark ships it natively:

* `MulticlassClassificationEvaluator(metricName="f1")` — the **weighted F1**: each class's F1 is weighted by its support (number of rows of that class). One native line of Spark.
* **Macro-F1** — the *unweighted* mean of the per-class F1 scores, which treats rare and frequent classes equally. On imbalanced data (here roughly 75% not haunted vs 25% haunted in the test split) it is usually the more honest metric. it is possible to compute them with the `fMeasureByLabel`/macro metric of its own (per-label metrics arrive with Spark 4.0). But for this notebook we will again compute it ourselves from the confusion matrix. The aggregation itself runs in Spark (`groupBy` + `count`); only the handful of confusion-matrix cells is collected to the driver — no pandas involved.

In [0]:
# Native, Spark-built-in weighted F1 (support-weighted average over classes)
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol=TARGET, predictionCol="prediction", metricName="f1"
)

def f1_macro(predictions, label_col=TARGET, prediction_col="prediction"):
    """Macro-averaged F1, derived from the confusion matrix with Spark
    aggregations. The groupBy/count runs distributed; only the few
    confusion-matrix cells are collected to the driver."""
    cm = predictions.groupBy(label_col, prediction_col).count().collect()
    counts = {(row[0], row[1]): row[2] for row in cm}
    labels = sorted({key[0] for key in counts} | {key[1] for key in counts})

    f1_scores = []
    for label in labels:
        tp = counts.get((label, label), 0)
        fp = sum(c for (l, p), c in counts.items() if p == label and l != label)
        fn = sum(c for (l, p), c in counts.items() if l == label and p != label)
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall    = tp / (tp + fn) if (tp + fn) else 0.0
        f1_scores.append(
            2 * precision * recall / (precision + recall)
            if (precision + recall) else 0.0
        )
    return sum(f1_scores) / len(f1_scores)

f1_weighted = f1_evaluator.evaluate(results)
macro_f1    = f1_macro(results)

print(f"Pipeline LR — Test weighted F1 (native Spark): {f1_weighted:.4f}")
print(f"Pipeline LR — Test macro-F1 (helper):          {macro_f1:.4f}")

## 5. Saving and Loading a Pipeline

A fitted `PipelineModel` can be written to disk and reloaded later. This is how a trained model moves from a training notebook to a production scoring job — the loaded model is a fully functional transformer that applies all stages to new data with a single `.transform()` call, with no retraining.

In [0]:
MODEL_PATH = "/Volumes/workspace/default/bda_week11/hmspia_pipeline_lr"

# Save the trained pipeline
pipeline_model.write().overwrite().save(MODEL_PATH)

# Load the saved pipeline
from pyspark.ml import PipelineModel

loaded_model = PipelineModel.load(MODEL_PATH)

# Evaluate the loaded model — native weighted F1
f1_loaded = evaluator.evaluate(loaded_model.transform(test))

print(f"F1 from loaded pipeline: {f1_loaded:.4f} (should match {f1_pipeline:.4f})")

## 6. Experiment Tracking with MLflow

In [0]:
EXPERIMENT_NAME = "/Users/dcaridade@novaims.unl.pt/BigDataAnalysis2627/weeks/week11/lab11_pipelines_tuning"
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name="baseline-pipeline-logistic-regression") as run:
    # Params: how the model was built (pipeline + algorithm settings)
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("train_fraction", 0.7)
    mlflow.log_param("split_seed", 753)
    mlflow.log_param("regParam", lr_model.getRegParam())
    mlflow.log_param("elasticNetParam", lr_model.getElasticNetParam())
    mlflow.log_param("imputer_strategy", imputer.getStrategy())

    # Metrics: training AND held-out test performance of the fitted pipeline.
    # The gap between train and test is a quick overfitting diagnostic.
    train_results = pipeline_model.transform(training)
    mlflow.log_metric("train_precision", precision_evaluator.evaluate(train_results))
    mlflow.log_metric("train_recall",    recall_evaluator.evaluate(train_results))
    mlflow.log_metric("train_f1",        evaluator.evaluate(train_results))
    mlflow.log_metric("test_precision",  precision_evaluator.evaluate(results))
    mlflow.log_metric("test_recall",     recall_evaluator.evaluate(results))
    mlflow.log_metric("test_f1",         f1_weighted)
    mlflow.log_metric("test_f1_macro",   macro_f1)

    # The pipeline itself was persisted in Section 5; record where it lives
    mlflow.set_tag("model_path", MODEL_PATH)

    print(f"Baseline run logged: {run.info.run_id}")

## 7. Hyperparameter Tuning with CrossValidator

`CrossValidator` wraps a pipeline (or any estimator) and a grid of hyperparameter values, refitting the pipeline once for each combination across K folds of the training data. It returns the best model according to the evaluator metric — the pipeline refit on the full training set with the winning hyperparameters.

`ParamGridBuilder` constructs the grid. Each `.addGrid(param, values)` call adds one hyperparameter with a list of candidate values; the builder takes the Cartesian product across all added parameters.

### The leakage trap in cross-validation

Cross-validation only produces an honest estimate if **every** learning step — preprocessing included — is refit on each fold's training part alone. There are two ways to wire the same components, and only one of them is correct:

```python
# WRONG — preprocessing fitted once, on the FULL training set
preprocess_model = preprocess_pipeline.fit(training)   # sees every fold's data
prepared         = preprocess_model.transform(training)
cv = CrossValidator(estimator=lr, ...)                  # folds of *prepared* data
# The imputation means, scaling bounds and category mappings now carry
# information from every validation fold -> optimistic, biased scores.

# CORRECT — preprocessing nested INSIDE the estimator, refit on every fold
cv = CrossValidator(
    estimator=Pipeline(stages=[preprocess_pipeline, lr]),
    ...
)
# For every fold AND every parameter combination, CrossValidator calls
# .fit() on the fold's training part only, so the Imputer, MinMaxScaler and
# StringIndexer are relearned without ever seeing the validation fold.
```

With the pipeline nested inside the `CrossValidator`, each of the K folds × P parameter combinations triggers a complete refit of the preprocessing stages and the classifier. That is the price of an unbiased estimate — and it is exactly what we want: transformations never carry information from the validation fold into the model, and the test set is not touched at all until the final evaluation.

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder


# ============================================================
# 1. Define the preprocessing pipeline
# ============================================================

preprocess_pipeline = Pipeline(stages=[
    indexer,
    encoder,
    imputer,
    numeric_assembler,
    scaler,
    feature_assembler
])


# ============================================================
# 2. Initialize Logistic Regression
# ============================================================

lr_crossval = LogisticRegression(
    featuresCol="features",
    labelCol=TARGET
)


# ============================================================
# 3. Create the full pipeline
# ============================================================

crossval_pipeline = Pipeline(stages=[
    preprocess_pipeline,
    lr_crossval
])


# ============================================================
# 4. Define the parameter grid
# ============================================================

param_grid = (
    ParamGridBuilder()
    .addGrid(lr_crossval.regParam, [0.01, 0.1])
    .addGrid(lr_crossval.elasticNetParam, [0.0, 1.0])
    .build()
)


# ============================================================
# 5. Configure cross-validation
# ============================================================

cross_validator = CrossValidator(
    estimator=crossval_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    seed=753
)


# ============================================================
# 6. Fit the cross-validator
# ============================================================

cross_validator_model = cross_validator.fit(training)


# ============================================================
# 7. Evaluate the best model on the test set
# ============================================================

crossval_results = cross_validator_model.transform(test)

f1_cv = evaluator.evaluate(crossval_results)

print(f"Cross-validated model test weighted F1: {f1_cv:.4f}")
print(
    "Average validation weighted F1 per parameter combination:",
    [round(m, 4) for m in cross_validator_model.avgMetrics]
)


# ============================================================
# 8. Extract the best Logistic Regression model
# ============================================================

best_pipeline = cross_validator_model.bestModel

# Outer pipeline:
# Stage 0 = fitted preprocessing pipeline
# Stage 1 = fitted Logistic Regression model
best_preprocess = best_pipeline.stages[0]
best_lr = best_pipeline.stages[1]

print(f"Best regParam:        {best_lr.getRegParam()}")
print(f"Best elasticNetParam: {best_lr.getElasticNetParam()}")

### 7.1 Storing the grid search in MLflow

The `avgMetrics` array of the fitted `CrossValidatorModel` holds the average validation score of every parameter combination, in the same order as `getEstimatorParamMaps()`. We log **one nested child run per combination** under a single parent run: the Experiments sidebar can then sort and compare candidates by their validation F1, while the parent run records the search configuration, the winning combination and the final training and test metrics.

This is the grid-search equivalent of a lab notebook — nothing is lost, and every candidate's hyperparameters are stored in MLflow.

In [0]:
param_maps  = cross_validator.getEstimatorParamMaps()
avg_metrics = cross_validator_model.avgMetrics
best_idx    = max(range(len(avg_metrics)), key=lambda i: avg_metrics[i])

with mlflow.start_run(run_name="grid-search-logistic-regression") as gs_run:
    # Parent run: the search configuration and the best result
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("num_folds", cross_validator.getNumFolds())
    mlflow.log_param("grid_size", len(param_maps))

    # One nested child run per parameter combination, so every candidate's
    # hyperparameters and average validation score are stored in MLflow
    for i, (params, avg_metric) in enumerate(zip(param_maps, avg_metrics)):
        with mlflow.start_run(run_name=f"combo-{i}", nested=True):
            for param, value in params.items():
                mlflow.log_param(param.name, value)
            mlflow.log_metric("avg_f1_weighted", avg_metric)

    # Best combination and its training + test performance, on the parent run
    for param, value in param_maps[best_idx].items():
        mlflow.log_param(f"best_{param.name}", value)
    crossval_train_results = cross_validator_model.transform(training)
    mlflow.log_metric("train_precision", precision_evaluator.evaluate(crossval_train_results))
    mlflow.log_metric("train_recall",    recall_evaluator.evaluate(crossval_train_results))
    mlflow.log_metric("train_f1",        f1_evaluator.evaluate(crossval_train_results))
    mlflow.log_metric("test_precision",  precision_evaluator.evaluate(crossval_results))
    mlflow.log_metric("test_recall",     recall_evaluator.evaluate(crossval_results))
    mlflow.log_metric("test_f1",         f1_cv)
    mlflow.log_metric("test_f1_macro",   f1_macro(crossval_results))

    print(f"Best combination (index {best_idx}): "
          f"regParam={param_maps[best_idx][lr_crossval.regParam]}, "
          f"elasticNetParam={param_maps[best_idx][lr_crossval.elasticNetParam]}")
    print(f"Grid search logged under parent run: {gs_run.info.run_id}")

### 7.2 Exercise — Grid search a Random Forest

**Your turn.** Repeat the grid-search workflow of Section 7, this time for a `RandomForestClassifier`:

1. Grid: `numTrees ∈ {20, 50}` and `maxDepth ∈ {3, 7}`.
2. **Leakage-safe wiring:** the estimator you pass to `CrossValidator` must be the *nested pipeline* `Pipeline(stages=[preprocess_pipeline, rf])` — never a bare `rf` on already-transformed data.
3. Evaluator: the **native Spark F1** — `MulticlassClassificationEvaluator` with `metricName="f1"` (weighted F1, the `f1_evaluator` from Section 4.4).
4. 3 folds, `seed=753`, fitted on `training` only; evaluate the best model on `test` exactly once.
5. Report test precision, recall, weighted F1 (native Spark) and macro-F1 (the `f1_macro` helper).

Before running, double-check: is the preprocessing refit inside every fold? Is the test set touched only once, at the very end?

In [0]:
# 1. Instantiate the RandomForestClassifier
rf = RandomForestClassifier(featuresCol="features", labelCol=TARGET, seed=753)

# 2. Build the leakage-safe estimator:
rf_pipeline = Pipeline(stages=[preprocess_pipeline, rf])

# 3. Build the grid with ParamGridBuilder:
rf_param_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [20, 50])
    .addGrid(rf.maxDepth, [3, 7])
    .build()
)

# 4. Create the CrossValidator with the native F1 evaluator
rf_cross_validator = CrossValidator(
    estimator          = rf_pipeline,
    estimatorParamMaps = rf_param_grid,
    evaluator          = f1_evaluator,
    numFolds           = 3,
    seed               = 753,
)

# 5. Fit on `training` only.
rf_cv_model = rf_cross_validator.fit(training)

# 6. Transform `test` once with the best model and report precision,
rf_train_predictions = rf_cv_model.transform(training)   # training metrics
rf_predictions       = rf_cv_model.transform(test)        # test set, used once

prec_rf_cv      = precision_evaluator.evaluate(rf_predictions)   # native weighted precision
rec_rf_cv       = recall_evaluator.evaluate(rf_predictions)      # native weighted recall
f1_rf_cv        = f1_evaluator.evaluate(rf_predictions)         # native weighted F1
f1_macro_rf_cv  = f1_macro(rf_predictions)                      # macro-F1 helper

best_rf = rf_cv_model.bestModel.stages[-1]
print(f"Best numTrees: {best_rf.getOrDefault('numTrees')}, best maxDepth: {best_rf.getOrDefault('maxDepth')}")
print(f"RF grid-search — test precision (native Spark): {prec_rf_cv:.4f}")
print(f"RF grid-search — test recall    (native Spark): {rec_rf_cv:.4f}")
print(f"RF grid-search — test weighted F1 (native Spark): {f1_rf_cv:.4f}")
print(f"RF grid-search — test macro-F1 (helper):          {f1_macro_rf_cv:.4f}")

# 7. Log every combination and the best result (training AND test
rf_param_maps  = rf_cross_validator.getEstimatorParamMaps()
rf_avg_metrics = rf_cv_model.avgMetrics
rf_best_idx    = max(range(len(rf_avg_metrics)), key=lambda i: rf_avg_metrics[i])

with mlflow.start_run(run_name="grid-search-random-forest") as rf_parent:
    mlflow.log_param("model", "RandomForestClassifier")
    mlflow.log_param("num_folds", 3)
    mlflow.log_param("grid_size", len(rf_param_maps))

    for i, (params, avg_metric) in enumerate(zip(rf_param_maps, rf_avg_metrics)):
        with mlflow.start_run(run_name=f"combo-{i}", nested=True):
            for param, value in params.items():
                mlflow.log_param(param.name, value)
            mlflow.log_metric("avg_f1_weighted", avg_metric)

    for param, value in rf_param_maps[rf_best_idx].items():
        mlflow.log_param(f"best_{param.name}", value)
    mlflow.log_metric("train_precision", precision_evaluator.evaluate(rf_train_predictions))
    mlflow.log_metric("train_recall",    recall_evaluator.evaluate(rf_train_predictions))
    mlflow.log_metric("train_f1",        f1_evaluator.evaluate(rf_train_predictions))
    mlflow.log_metric("test_precision", prec_rf_cv)
    mlflow.log_metric("test_recall",    rec_rf_cv)
    mlflow.log_metric("test_f1",         f1_rf_cv)
    mlflow.log_metric("test_f1_macro",  f1_macro_rf_cv)

## 8. Word2Vec for Text Features

In Part 1 Section 8.3 we used TF-IDF to incorporate free-text columns. TF-IDF is a bag-of-words approach: it counts words but ignores their meaning and order. **Word2Vec** is a shallow neural network that instead learns a dense vector representation (an *embedding*) for each word such that semantically similar words sit close together in vector space.

Spark ML's `Word2Vec` takes a column of token arrays and returns, for each document, the **average** of its word vectors — a single dense vector regardless of document length. This is cruder than transformer-based embeddings but fast and entirely Spark-native.

Key parameters:
- `vectorSize` — dimensionality of each word embedding (default 100).
- `minCount` — words appearing fewer than this many times are ignored.
- `windowSize` — context window used during training.
- `inputCol` — must be an array of strings (i.e. already tokenized).
- `outputCol` — the resulting dense vector column.

### 8.1 Preparing text columns

Word2Vec expects tokenised input — an array of strings per row. We use `Tokenizer` (lowercases and splits on whitespace) and then concatenate all four text columns into a single token array before embedding.

In [0]:
# These are the free-text columns of our dataset
TEXT_COLS = [
    "estate_agent_euphemism", "cat_behaviour",
    "vicar_response", "most_recent_excuse",
    "previous_owners_fate", "archaeology_report_status"
]

# Fill nulls so Tokenizer doesn't fail, then concatenate into one string
data_text = properties
for text_col in TEXT_COLS:
    data_text = data_text.fillna("", subset=[text_col])

data_text = data_text.withColumn(
    "all_text",
    concat_ws(" ", *[col(text_col) for text_col in TEXT_COLS])
)

tokenizer = Tokenizer(inputCol="all_text", outputCol="text_tokens")
data_text = tokenizer.transform(data_text)
data_text.select("text_tokens").show(3, truncate=80)

### 8.2 Fitting Word2Vec as a standalone estimator

Before plugging Word2Vec into a pipeline, we inspect it in isolation to understand what it learns. After fitting we can query the model for the words most similar to a given word — a useful sanity check.

In [0]:
word2vec = Word2Vec(
    vectorSize = 150,
    minCount   = 3,
    windowSize = 5,
    inputCol   = "text_tokens",
    outputCol  = "text_embedding",
    seed       = 753,
)

word2vec_model = word2vec.fit(data_text)

# Inspect the learned vocabulary — find words similar to 'vicar'
word2vec_model.findSynonyms("vicar", 5).show(truncate=False)

In [0]:
# Transform: each document becomes the mean of its word vectors
data_text = word2vec_model.transform(data_text)
data_text.select("text_embedding").show(3, truncate=80)

### 8.2.1 Which text column matters most?

Before building the full pipeline, it is worth asking whether all four text columns carry equal predictive weight, or whether one dominates.

**The approach.** For each column *in isolation* we:
1. Tokenise it.
2. Fit a Word2Vec model on the training split of that column alone.
3. Transform both splits to get a document-level embedding (the mean of its token vectors).
4. Assemble that single embedding into `features` and train a Logistic Regression.
5. Record precision, recall and F1 (native Spark evaluators) on the test split.

**What this is and isn't.** This is a *marginal contribution* screen — each column is tested in isolation, so interaction effects are ignored. It is a fast, interpretable guide for deciding which columns to invest in further (e.g. with a larger `vectorSize`, more preprocessing, or a dedicated PCA stage). It does **not** tell you the column's contribution inside a joint model, where a weaker column may still add signal that the others lack.

**TODO:** Complete the loop. For each `text_col` in `TEXT_COLS`:
1. Tokenise it with `Tokenizer` into `"single_column_tokens"`.
2. Fit `Word2Vec` (vectorSize=150, minCount=2, seed=753) on the training split.
3. Transform both splits; assemble the embedding into `"features"` with `VectorAssembler`.

Which column wins? Does the result match your intuition about the dataset?

In [0]:
f1_scores_by_column = {}

for text_col in TEXT_COLS:
    # 1. Handle missing text
    column_data = properties.fillna("", subset=[text_col])

    # 2. Tokenize the selected text column
    tokenizer = Tokenizer(
        inputCol=text_col,
        outputCol="single_column_tokens"
    )
    column_data = tokenizer.transform(column_data)

    # 3. Split into training and test sets
    train_data, test_data = column_data.randomSplit(
        [0.7, 0.3],
        seed=753
    )

    # 4. Train Word2Vec using training data only
    word2vec = Word2Vec(
        vectorSize=150,
        minCount=2,
        inputCol="single_column_tokens",
        outputCol="column_embedding",
        seed=753
    )

    column_w2v_model = word2vec.fit(train_data)

    # 5. Generate embeddings
    train_embedded = column_w2v_model.transform(train_data)
    test_embedded = column_w2v_model.transform(test_data)

    # 6. Assemble feature vectors
    assembler = VectorAssembler(
        inputCols=["column_embedding"],
        outputCol="features"
    )

    train_embedded = assembler.transform(train_embedded)
    test_embedded = assembler.transform(test_embedded)

    # 7. Train Logistic Regression
    logistic_regression = LogisticRegression(
        featuresCol="features",
        labelCol=TARGET
    )

    # NOTE: a distinct name, so this per-column model does NOT shadow
    # lr_model — the fitted pipeline model from Section 4
    column_lr_model = logistic_regression.fit(train_embedded)

    # 8. Evaluate on the test set — native precision, recall and F1
    predictions = column_lr_model.transform(test_embedded)

    precision_score = precision_evaluator.evaluate(predictions)
    recall_score    = recall_evaluator.evaluate(predictions)
    f1_score        = f1_evaluator.evaluate(predictions)
    f1_scores_by_column[text_col] = round(f1_score, 4)

    print(f"Processed {text_col}: precision = {precision_score:.4f}, "
          f"recall = {recall_score:.4f}, F1 = {f1_score:.4f}")


# 9. Rank text columns by F1
print("\nF1 scores by text column:")

for text_col, f1_score in sorted(
    f1_scores_by_column.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{text_col:35s}  F1 = {f1_score:.4f}")

### 8.2.2 Vector arithmetic: exploring the embedding space

Word2Vec encodes semantic relationships as **directions** in vector space. The canonical demonstration is an analogy task: if the model has learned a consistent direction for a relationship, arithmetic on vectors should land near the expected answer.

A well-known example from generic corpora is the *country → capital* relationship:

> `vec(France) − vec(Paris) ≈ vec(Germany) − vec(Berlin)`  
> equivalently: `vec(Germany) − vec(Berlin) + vec(Paris) ≈ vec(France)`

Our corpus has its own consistent relationships across four distinct registers — the estate agent's euphemisms, the vicar's responses, the cat's behaviour, and the excuses offered by previous occupants. Good analogies exploit the *contrast between registers*:

| Column | Analogy | Intuition |
|---|---|---|
| Estate agent euphemism | `poltergeist − haunted + potential ≈ ?` | What does an agent call a poltergeist? |
| Owner Excuse | `ghost − haunted + pipes ≈ ?` | The same phenomenon, rationalised as a plumbing issue |
| Cat behaviour | `cat − ghost + interest ≈ ?` | If not a ghost, what is the resident cat then interested in? |

The arithmetic works as follows: given `positive` words (add) and `negative` words (subtract), compute a result vector, then call `findSynonyms` with it — `findSynonyms` accepts either a word **string** *or* a raw `DenseVector`.

> **Note on expectations.** The corpus is not enormous, so not every analogy will resolve cleanly — that is a valid and interesting result in itself. If a query word is missing from the vocabulary, `analogy()` will tell you. Try browsing `vocab.keys()` to find words that are actually present before designing your own experiments.


In [0]:
# Retrieve all word vectors from the trained Word2Vec model
vocabulary_df = word2vec_model.getVectors()

def get_vector(word):
    # Filter the vocabulary for a specific word and extract its vector
    row = vocabulary_df.filter(col("word") == word).first()
    return row["vector"] if row else None

In [0]:

from pyspark.sql.functions import col, lit
from pyspark.ml.linalg import Vectors

def analogy(positive, negative=None, k=5):
    negative = negative or []

    # 1. Assign weights to the input words
    weights = (
        [(w, 1.0) for w in positive]
        + [(w, -1.0) for w in negative]
    )

    weights_df = spark.createDataFrame(
        weights, ["word", "weight"]
    )

    # 2. Find which input words exist in the Word2Vec vocabulary
    matched = vocabulary_df.join(
        weights_df, on="word", how="inner"
    )

    matched_words = {
        row["word"] for row in matched.select("word").distinct().collect()
    }

    missing_words = set(positive + negative) - matched_words

    if missing_words:
        print("Words not found in vocabulary:", sorted(missing_words))

    if not matched_words:
        print("None of the input words are in the Word2Vec vocabulary.")
        print("Try words that appear in your training corpus.")
        return None

    # 3. Calculate the weighted vectors
    vectors = [
        (row["vector"], row["weight"])
        for row in matched.select("vector", "weight").collect()
    ]

    # 4. Sum the weighted vectors locally
    vector_size = len(vectors[0][0])

    result = [0.0] * vector_size

    for vector, weight in vectors:
        for i, value in enumerate(vector):
            result[i] += weight * value

    query_vector = Vectors.dense(result)

    # 5. Exclude input words from the results
    exclude = list(matched_words)

    # 6. Find similar words
    synonyms = word2vec_model.findSynonyms(
        query_vector, k + len(exclude)
    )

    return (
        synonyms
        .filter(~col("word").isin(exclude))
        .limit(k)
    )


# Example — what does an estate agent call a poltergeist?
result = analogy(
    positive=["poltergeist", "potential"],
    negative=["haunted"]
)

if result is not None:
    result.show(truncate=False)

In [0]:
# Experiment B — supernatural phenomenon → mundane excuse
analogy(positive=["ghost", "pipes"], negative=["haunted"]).show()


In [0]:
# Experiment C — feline alarm → feline curiosity
analogy(positive=["cat", "interest"], negative=["ghost"]).show()

In [0]:
def cosine_sim(w1, w2):
    """Spark-native similarity using **0.5 for square root to avoid math import."""
    v1, v2 = get_vector(w1), get_vector(w2)
    if not v1 or not v2: return 0.0
    
    dot = v1.dot(v2)
    norm1 = v1.dot(v1)**0.5
    norm2 = v2.dot(v2)**0.5
    
    return float(dot / (norm1 * norm2)) if (norm1 * norm2) > 0 else 0.0

# Representative pairs from the six-column vocabulary, spanning the range
# of the measure: near 1 for words from the same register, low for words
# from unrelated registers — and spooky synonyms that are surprisingly
# far apart. 
pairs = [
    ("vicar",     "reverend"),
    ("destroyed", "replacement"),
    ("pending",   "withheld"),     
    ("ghost",     "poltergeist"),  
    ("haunted",   "pipes"),        
    ("vicar",     "cat"),           
]
for w1, w2 in pairs:
    print(f"cos({w1:12s}, {w2:12s}) = {cosine_sim(w1, w2):.4f}")

### 8.2.3 Clustering the vocabulary with KMeans

Vector arithmetic probes *specific* relationships. Clustering takes a broader view: it partitions the entire vocabulary into groups of semantically close words, without any prior hypothesis about which words belong together.

Spark ML's `KMeans` is an estimator that works on any `DenseVector` column — the same interface as the rest of the pipeline. We fit it directly on `vectors_df` (the vocabulary DataFrame returned by `getVectors()`).

The key design choice is **k** — the number of clusters. Use the **silhouette score** (`ClusteringEvaluator` with `metricName="silhouette"`, `distanceMeasure="cosine"`) to compare candidate values: +1 is perfect separation, 0 means overlapping, −1 means misclustered.

To inspect a fitted cluster, a useful summary is the **words closest to its centroid** — those are the most representative members. `cluster_label()` below does this: given a cluster id and a fitted model, it computes cosine distances from all member words to the centroid and returns the top-n closest.


In [0]:
# Prepare data for clustering by renaming the vector column to 'features'
clustering_df = vocabulary_df.withColumnRenamed("vector", "features")

def cluster_label(cluster_id, model, data, n=8):
    # 1. Get the centroid for the specific cluster
    centroid = model.clusterCenters()[cluster_id]
    # 2. Collect all words assigned to this cluster
    members = [r for r in data.collect() if r["prediction"] == cluster_id]
    
    def sim(v1, v2):
        # Manual cosine similarity: (A . B) / (||A|| * ||B||)
        # Using **0.5 instead of math.sqrt
        dot = float(v1.dot(v2))
        norm1 = float(v1.dot(v1))**0.5
        norm2 = float(v2.dot(v2))**0.5
        return dot / (norm1 * norm2) if (norm1 * norm2) > 0 else 0.0
        
    # 3. Sort by proximity to centroid
    members.sort(key=lambda r: sim(r["features"], centroid), reverse=True)
    return [r["word"] for r in members[:n]]

# Sweep k values to find the best clustering configuration
clustering_evaluator = ClusteringEvaluator(featuresCol="features", metricName="silhouette", distanceMeasure="cosine")
scores = {}
for k in [5, 8, 12, 16, 20]:
    km = KMeans(featuresCol="features", k=k, seed=753).fit(clustering_df)
    scores[k] = clustering_evaluator.evaluate(km.transform(clustering_df))

# Display Silhouette scores for comparison
spark.createDataFrame(list(scores.items()), ["k", "silhouette"]).show()

# Finalize model with the best k found
best_k = max(scores, key=scores.get)
best_km = KMeans(featuresCol="features", k=best_k, seed=753).fit(clustering_df)

# NOTE: a distinct name, so this DataFrame does NOT shadow `results` —
# the pipeline test predictions used in Sections 4-7
cluster_results = best_km.transform(clustering_df)

# Display representative words for each cluster using a Spark DataFrame
summary = [(i, ", ".join(cluster_label(i, best_km, cluster_results))) for i in range(best_k)]
spark.createDataFrame(summary, ["cluster", "representative_words"]).show(truncate=False)


### 8.3 Word2Vec + PCA inside a pipeline

In Section 8.2.1 you ranked the four text columns by standalone F1. Use that result here: rather than concatenating all four columns blindly, choose which to include based on what you found.

**The dimensionality problem.** With `vectorSize=150`, the Word2Vec block contributes 150 dimensions to the final feature vector while the entire numeric and boolean block contributes roughly 20. A Random Forest sees far more candidate split points in the text block, which can crowd out the structured features even if they carry more signal.

**The fix: PCA.** We add a `PCA` stage immediately after `Word2Vec` to compress the 150-dimensional embedding down to `k` principal components — retaining most of the variance while bringing the text block back to a comparable scale. `PCA` is an estimator in `pyspark.ml.feature`: it fits on the training data to learn the principal directions, then transforms both splits.

In [0]:

def run_w2v_pipeline(df, text_col, label, pca_k=20, seed=753):
    """Run the Word2Vec + PCA + Random Forest workflow, logging training
    and test metrics to MLflow."""

    tokenizer = Tokenizer(
        inputCol=text_col,
        outputCol="tokens"
    )

    w2v = Word2Vec(
        vectorSize=150,
        minCount=3,
        inputCol="tokens",
        outputCol="w2v",
        seed=seed
    )

    pca = PCA(
        k=pca_k,
        inputCol="w2v",
        outputCol="pca"
    )

    # Assemble text, numeric, and boolean features
    assembler = VectorAssembler(
        inputCols=(
            ["numeric_scaled"]
            + bool_cols
            + ["phenomenon_vec", "ghost_vec", "pca"]
        ),
        outputCol="features"
    )

    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol=TARGET,
        numTrees=100,
        maxDepth=5,
        seed=seed
    )

    # Build the complete pipeline
    pipeline = Pipeline(stages=[
        indexer,
        encoder,
        imputer,
        numeric_assembler,
        scaler,
        tokenizer,
        w2v,
        pca,
        assembler,
        rf
    ])

    # Split the data WITHOUT caching
    train, test = df.randomSplit(
        [0.7, 0.3],
        seed=seed
    )

    # Train the pipeline
    model = pipeline.fit(train)

    # Predictions on BOTH the training and the test set
    train_predictions = model.transform(train)
    test_predictions  = model.transform(test)

    # Native Spark metrics: precision, recall and (weighted) F1
    metrics = {
        "train_precision": precision_evaluator.evaluate(train_predictions),
        "train_recall":    recall_evaluator.evaluate(train_predictions),
        "train_f1":        f1_evaluator.evaluate(train_predictions),
        "test_precision":  precision_evaluator.evaluate(test_predictions),
        "test_recall":     recall_evaluator.evaluate(test_predictions),
        "test_f1":         f1_evaluator.evaluate(test_predictions),
    }

    print(f"{label:40s} precision: {metrics['test_precision']:.4f}  "
          f"recall: {metrics['test_recall']:.4f}  F1: {metrics['test_f1']:.4f}")

    # Log training AND test metrics to MLflow — the gap between the two
    # is a quick overfitting diagnostic
    with mlflow.start_run(run_name=label):
        mlflow.log_param("model", "RandomForestClassifier + Word2Vec + PCA")
        mlflow.log_param("text_col", text_col)
        mlflow.log_param("pca_k", pca_k)
        mlflow.log_param("numTrees", 100)
        mlflow.log_param("maxDepth", 5)
        mlflow.log_param("split_seed", seed)
        for metric_name, value in metrics.items():
            mlflow.log_metric(metric_name, value)

    return metrics["test_f1"], model


# Run the pipeline
f1_w2v, w2v_pipeline_model = run_w2v_pipeline(
    properties,
    "estate_agent_euphemism",
    "RF + Word2Vec + PCA"
)

## 9. Final comparison

In [0]:
pipeline_results = [
    ("LR (pipeline)",       f1_pipeline),
    ("LR (cross-val)",      f1_cv),
    ("RF (grid-search)",    f1_rf_cv),
    ("RF + Word2Vec + PCA", f1_w2v),
]

summary_df = spark.createDataFrame(pipeline_results, ["Model", "F1"]) \
    .orderBy("F1", ascending=False)
summary_df.show(truncate=False)

names  = [r["Model"] for r in summary_df.collect()]
scores = [r["F1"]    for r in summary_df.collect()]

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(names, scores)
ax.set_xlim(0.5, 1.0)
ax.set_xlabel("F1 (weighted)")
ax.set_title("Pipeline Comparison — HMSPIA NAPR")
plt.tight_layout()
plt.show()